First we look at the process model 


In [1]:
import control as ct
import matplotlib.pyplot as plt
import numpy as np

T = 10 # samplingsfrekvens, once per 10 seconds

z = ct.tf('z')
G = T/(1-z**(-1))
plt.figure(1)
ct.pzmap(G)
plt.title("Pole/zero plot for plant")
plt.figure(2)
ct.bode_plot(G, display_margins=True)
plt.title("Bode plot of plant")
plt.show()


In [6]:
hr = np.linspace(0, 50, 10000)
Gi = 1/( 1 - z**(-1))
ct.rlocus(G*Gi , gains=hr)
plt.title("Possible conjugate pairs of poles using I-controller")
plt.show()


Now place the poles


In [3]:
Kp = .4
Ki = 0.5

Gpi = (Kp*(z-1) + Ki*z)/(z-1)
Gcl = ct.feedback(G*Gpi, 1)
print(Gcl)
poles = ct.poles(Gcl)
plt.figure(1)
ct.pzmap(Gcl)
plt.title("Pole/zero plot of PI-controller")
print(poles)
plt.figure(2)
ct.bode_plot(G*Gpi, display_margins=True)
plt.title("Bode plot of closed loop")
plt.show()


<TransferFunction>: sys[39]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = True
    9 z^2 - 4 z
  ----------------
  10 z^2 - 6 z + 1
[0.3+0.1j 0.3-0.1j]


Now find the Kp and Ki using pole placement


In [9]:
def design_discrete_pi(T, desired_poles):
    """
    Calculates Kp and Ki for a discrete PI controller 
    given a sampling time T and desired closed-loop poles.
    Assumes plant G(z) = (T*z) / (z-1)
    """
    desired_poly = np.poly(desired_poles)
    d1 = desired_poly[1]
    d0 = desired_poly[2]
    
    Kp = (-2 * d0 - d1) / (T * d0)
    Ki = (1 + d0 + d1) / (T * d0)
    
    return Kp, Ki


In [11]:
T = 10
desired_poles = [0.3 + 0.1j, 0.3 - 0.1j]
Kp_calc, Ki_calc = design_discrete_pi(T, desired_poles)

print(f"Calculated Kp: {Kp_calc:.4f}")
print(f"Calculated Ki: {Ki_calc:.4f}")

z = ct.tf('z')
G = T / (1 - z**(-1))
Gpi = (Kp_calc*(z-1) + Ki_calc*z) / (z-1)

Gcl = ct.feedback(G * Gpi, 1)

print(f"\nActual Closed-loop Poles for tau={T}:")
print(ct.poles(Gcl))

plt.figure(figsize=(6, 5))
ct.pzmap(Gcl)
plt.title(f"Correct Pole Placement via Diophantine")
plt.show()


Calculated Kp: 0.4000
Calculated Ki: 0.5000
Actual Closed-loop Poles for tau=10:
[0.3+0.1j 0.3-0.1j]
